# i01 · ¿Cuánto ha crecido la población de España y quién explica ese crecimiento?

- **Post:** `posts/i01-population-nationality/index.qmd`
- **Afirmación:** «España solo crece gracias a la inmigración.»
- **Pregunta a contrastar:** ¿cuánto ha crecido la población residente desde 2002 y qué parte de ese crecimiento corresponde a personas de nacionalidad extranjera o nacidas en el extranjero?
- **Datos:** INE (Cifras de Población hasta 2021, Estadística Continua de Población desde 2021)
  - descarga: `python src/download/download_ine_poblacion.py`
  - procesado: `python src/process/01_poblacion_nacionalidad.py`
  - salida: `data/processed/poblacion_nacionalidad.parquet`, `data/processed/poblacion_pais_nacimiento.parquet`

In [130]:
import os 
import polars as pl
from plotly.subplots import make_subplots

project_path =  os.path.join(os.getcwd(), '..')  
raw_data_folder_path = os.path.join(project_path, 'data', 'raw', 'ine_poblacion')

import sys
sys.path.insert(0, project_path)    # el project_path de tu primera celda
from src.utils.plotting import ACCENT, GREY, PALETTE, init_plotly, new_figure, add_source
init_plotly()

## 1. Datos crudos

TODO: abrir los ficheros de `RAW / "ine_poblacion"` y entender su estructura antes de escribir `src/process/01_poblacion_nacionalidad.py`.

- ¿Qué series trae la tabla? ¿Totales y desgloses mezclados?
- ¿Fechas: 1 de enero, 1 de julio, trimestrales?
- ¿Hay valores provisionales?

In [131]:
MESES = {
    "enero": "01", "febrero": "02", "marzo": "03", "abril": "04",
    "mayo": "05", "junio": "06", "julio": "07", "agosto": "08",
    "septiembre": "09", "octubre": "10", "noviembre": "11", "diciembre": "12",
}

def total_num() -> pl.Expr:
    # "47.615.033,864697" -> 47615033.864697 ; ".." u otros no numéricos -> null
    return (
        pl.col("Total")
        .str.replace_all(r"\.", "")
        .str.replace(",", ".")
        .cast(pl.Float64, strict=False).round().cast(pl.Int64)
    )

def periodo_fecha() -> pl.Expr:
    return pl.col("Periodo").str.replace_many(MESES).str.to_date("%d de %m de %Y")

def edad_num(col: str) -> pl.Expr:
    # "0 años" -> 0 ; "90 y más años" -> 90 ; "Todas las edades" -> null
    return pl.col(col).str.extract(r"^(\d+)").cast(pl.Int8).alias("edad_num")

def provincia_cols() -> list[pl.Expr]:
    p = pl.col("Provincias")
    return [
        p.str.extract(r"^(\d{2})").alias("provincia_codigo"),
        p.str.replace(r"^\d{2}\s+", "").alias("provincia_nombre"),
    ]

def procesar(path, edad_col: str | None = None, provincial: bool = True) -> pl.DataFrame:
    exprs = [total_num(), periodo_fecha()]
    if provincial:
        exprs += provincia_cols()
    if edad_col:
        exprs.append(edad_num(edad_col))
    lf = pl.scan_csv(path, separator=";").with_columns(exprs)
    if provincial:
        lf = lf.drop("Provincias")
    return lf.collect()

raw_file_paths = {}
for file in os.listdir(raw_data_folder_path):
    raw_file_paths[file.split('.')[0]] = os.path.join(raw_data_folder_path, file)

df1 = procesar(raw_file_paths["56938"], edad_col="Edad simple", provincial=False)
df2 = procesar(raw_file_paths["60129"], edad_col="Grupo quinquenal de edad")
df3 = procesar(raw_file_paths["60130"], edad_col="Grupo quinquenal de edad")
df4 = procesar(raw_file_paths["9691"])

## 2. Exploración

### Q1.1: Cómo ha crecido la población? A qué ritmos? En qué periodos? --> A nivel Nacional

In [132]:
def es(x, d=2):
    """Número con formato español: 1.234,56"""
    return f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")

serie_total = (
    df1
    .filter(
        pl.col("Sexo") == "Total",
        pl.col("Edad simple") == "Todas las edades",
        pl.col("Nacionalidad") == "Total",
        pl.col("País de nacimiento") == "Total",
        pl.col("Total").is_not_null(),
    )
    .select(fecha=pl.col("Periodo"), poblacion=pl.col("Total"))
    .sort("fecha")
)

seria_total_anual_nacional = (
    serie_total
    .filter(pl.col("fecha").dt.month() == 1, pl.col("fecha").dt.day() == 1)
    .sort("fecha")
    .with_columns(
        var_abs=pl.col("poblacion").diff(),
        var_pct=100 * pl.col("poblacion").pct_change(),
    )
)

limites = [2002, 2008, 2014, 2020, 2025]
periodos = pl.DataFrame({"ini": limites[:-1], "fin": limites[1:]})

poblacion = seria_total_anual_nacional.select(año=pl.col("fecha").dt.year(), poblacion=pl.col("poblacion"))

resumen = (
    periodos
    .join(pob.rename({"año": "ini", "poblacion": "pob_ini"}), on="ini")
    .join(pob.rename({"año": "fin", "poblacion": "pob_fin"}), on="fin")
    .with_columns(años=pl.col("fin") - pl.col("ini"), var_abs=pl.col("pob_fin") - pl.col("pob_ini"))
    .with_columns(
        var_pct_total=100 * (pl.col("pob_fin") / pl.col("pob_ini") - 1),
        media_anual=pl.col("var_abs") / pl.col("años"),
        tasa_anual=100 * ((pl.col("pob_fin") / pl.col("pob_ini")) ** (1 / pl.col("años")) - 1),
    )
    .with_columns(periodo=pl.format("{}–{}", pl.col("ini"), pl.col("fin")))
).sort("ini")

crecimiento = (
    seria_total_anual_nacional
    .drop_nulls("var_abs")
    .with_columns(
        año=pl.col("fecha").dt.year(),
        tramo=pl.format("{} → {}", pl.col("fecha").dt.year() - 1, pl.col("fecha").dt.year()),
    )
)

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_bar(
    x=crecimiento["año"], y=crecimiento["var_abs"],
    name="Variación respecto al año anterior (eje izq.)", marker_color=ACCENT, opacity=0.85,
    customdata=crecimiento.select("var_pct", "tramo").rows(),
    hovertemplate="%{customdata[1]}<br>%{y:,.0f} personas<br>%{customdata[0]:.2f} %<extra>Variación</extra>",
    secondary_y=False,
)
fig.add_scatter(
    x=poblacion["año"], y=poblacion["poblacion"],
    name="Población a 1 de enero (eje der.)", mode="lines+markers",
    line=dict(color=PALETTE[1], width=2.5), marker=dict(size=5),
    hovertemplate="%{y:,.0f}<extra>Población</extra>",
    secondary_y=True,
)

for i, r in enumerate(resumen.iter_rows(named=True)):
    x0, x1 = r["ini"] + 0.5, r["fin"] + 0.5      # cada banda contiene las barras ini+1 … fin

    # Banda de fondo en periodos alternos
    if i % 2 == 0:
        fig.add_vrect(x0=x0, x1=x1, fillcolor="rgba(127,127,127,0.10)", line_width=0, layer="below")

    # Media anual del periodo, en el eje izquierdo (comparable con las barras)
    fig.add_scatter(
        x=[x0, x1], y=[r["media_anual"]] * 2, mode="lines",
        line=dict(color="#0d9488", width=2.5, dash="dash"),
        name="Media anual del periodo (eje izq.)", legendgroup="media", showlegend=(i == 0),
        hoverinfo="skip", secondary_y=False,
    )

    # Etiqueta sobre la banda
    fig.add_annotation(
        x=(x0 + x1) / 2, y=1.02, xref="x", yref="paper", yanchor="bottom", showarrow=False,
        text=f"<b>{r['periodo']}</b><br>{es(r['tasa_anual'])} % anual<br>{es(r['media_anual'] / 1000, 0)} mil/año",
        font=dict(size=12),
    )

fig.update_xaxes(range=[2001.5, 2025.5], dtick=2)
add_source(fig, "Fuente: INE, Estadística Continua de Población. Elaboración propia.")
fig.update_layout(
    margin=dict(t=110, b=130, r=70),
    legend=dict(orientation="h", x=0, y=-0.24, yanchor="top"),   # la leyenda baja para dejar sitio a las etiquetas
)
fig.show()

In [133]:
seria_total_anual_nacional

fecha,poblacion,var_abs,var_pct
date,i64,i64,f64
2002-01-01,41035271,null,null
2003-01-01,41827836,792565,1.931424
2004-01-01,42547454,719618,1.720428
2005-01-01,43296335,748881,1.760108
2006-01-01,44009969,713634,1.648255
…,…,…,…
2021-01-01,47400798,82748,0.174876
2022-01-01,47486727,85929,0.181282
2023-01-01,48085361,598634,1.260634


In [134]:
resumen

ini,fin,pob_ini,pob_fin,años,var_abs,var_pct_total,media_anual,tasa_anual,periodo
i64,i64,i64,i64,i64,i64,f64,f64,f64,str
2002,2008,41035271,45668938,6,4633667,11.291913,772277.833333,1.799099,"""2002–2008"""
2008,2014,45668938,46495744,6,826806,1.810434,137801.0,0.299488,"""2008–2014"""
2014,2020,46495744,47318050,6,822306,1.768562,137051.0,0.292611,"""2014–2020"""
2020,2025,47318050,49128297,5,1810247,3.825701,362049.4,0.753693,"""2020–2025"""


### Q1.2: Cómo se distribuye el creciemiento poblacional en función de la nacionalidad y del pais de nacimiento? --> A nivel Nacional

In [139]:
base = (
    df1
    .filter(
        pl.col("Sexo") == "Total",
        pl.col("Edad simple") == "Todas las edades",
        pl.col("Total").is_not_null(),
        pl.col("Periodo").dt.month() == 1, pl.col("Periodo").dt.day() == 1,
    )
    .select(año=pl.col("Periodo").dt.year(), nacionalidad=pl.col("Nacionalidad"),
            pais=pl.col("País de nacimiento"), poblacion=pl.col("Total"))
)

# A. Por nacionalidad
por_nacionalidad = (
    base.filter(pl.col("nacionalidad").is_in(["Española", "Extranjera"]), pl.col("pais") == "Total")
    .select("año", grupo=pl.col("nacionalidad"), poblacion=pl.col("poblacion"))
)

# B. Por país de nacimiento (España / extranjero)
por_nacimiento = (
    base.filter(pl.col("nacionalidad") == "Total", pl.col("pais").is_in(["España", "Extranjero"]))
    .select("año",
            grupo=pl.col("pais").replace_strict({"España": "Nacidos en España", "Extranjero": "Nacidos en el extranjero"}),
            poblacion=pl.col("poblacion"))
)

# C. Los cuatro grupos (cruce)
MAP4 = {
    "Española | España":      "Españoles nacidos en España",
    "Española | Extranjero":  "Españoles nacidos en el extranjero",
    "Extranjera | España":    "Extranjeros nacidos en España",
    "Extranjera | Extranjero": "Extranjeros nacidos en el extranjero",
}
cuatro_grupos = (
    base.filter(pl.col("nacionalidad").is_in(["Española", "Extranjera"]), pl.col("pais").is_in(["España", "Extranjero"]))
    .select("año",
            grupo=pl.concat_str(["nacionalidad", "pais"], separator=" | ").replace_strict(MAP4),
            poblacion=pl.col("poblacion"))
)

def resumen_periodos(anual, limites):
    periodos = pl.DataFrame({"ini": limites[:-1], "fin": limites[1:]})
    ini = anual.rename({"año": "ini", "poblacion": "pob_ini"})
    fin = anual.rename({"año": "fin", "poblacion": "pob_fin"})
    return (
        periodos
        .join(ini, on="ini")
        .join(fin, on=["fin", "grupo"])
        .with_columns(años=pl.col("fin") - pl.col("ini"), var_abs=pl.col("pob_fin") - pl.col("pob_ini"))
        .with_columns(
            media_anual=pl.col("var_abs") / pl.col("años"),
            tasa_anual=100 * ((pl.col("pob_fin") / pl.col("pob_ini")) ** (1 / pl.col("años")) - 1),
            contribucion=100 * pl.col("var_abs") / pl.col("var_abs").sum().over("ini"),   # % del Δ total del periodo
            periodo=pl.format("{}–{}", pl.col("ini"), pl.col("fin")),
        )
        .sort("ini", "grupo")
    )

res_A = resumen_periodos(por_nacionalidad, limites)
res_B = resumen_periodos(por_nacimiento, limites)
res_C = resumen_periodos(cuatro_grupos, limites)

In [140]:
res_A

ini,fin,grupo,pob_ini,pob_fin,años,var_abs,media_anual,tasa_anual,contribucion,periodo
i64,i64,str,i64,i64,i64,i64,f64,f64,f64,str
2002,2008,"""Española""",39297299,40582643,6,1285344,214224.0,0.537852,27.73924,"""2002–2008"""
2002,2008,"""Extranjera""",1737972,5086295,6,3348323,558053.833333,19.598701,72.26076,"""2002–2008"""
2008,2014,"""Española""",40582643,41819392,6,1236749,206124.833333,0.501582,149.581522,"""2008–2014"""
2008,2014,"""Extranjera""",5086295,4676352,6,-409943,-68323.833333,-1.390761,-49.581522,"""2008–2014"""
2014,2020,"""Española""",41819392,42076772,6,257380,42896.666667,0.102314,31.299784,"""2014–2020"""
2014,2020,"""Extranjera""",4676352,5241278,6,564926,94154.333333,1.918964,68.700216,"""2014–2020"""
2020,2025,"""Española""",42076772,42216326,5,139554,27910.8,0.066245,7.709114,"""2020–2025"""
2020,2025,"""Extranjera""",5241278,6911971,5,1670693,334138.6,5.689767,92.290886,"""2020–2025"""


In [141]:
res_B

ini,fin,grupo,pob_ini,pob_fin,años,var_abs,media_anual,tasa_anual,contribucion,periodo
i64,i64,str,i64,i64,i64,i64,f64,f64,f64,str
2002,2008,"""Nacidos en España""",38701173,39790019,6,1088846,181474.333333,0.463507,23.498581,"""2002–2008"""
2002,2008,"""Nacidos en el extranjero""",2334098,5878919,6,3544821,590803.5,16.644177,76.501419,"""2002–2008"""
2008,2014,"""Nacidos en España""",39790019,40542974,6,752955,125492.5,0.312929,91.067917,"""2008–2014"""
2008,2014,"""Nacidos en el extranjero""",5878919,5952770,6,73851,12308.5,0.208279,8.932083,"""2008–2014"""
2014,2020,"""Nacidos en España""",40542974,40303297,6,-239677,-39946.166667,-0.098772,-29.146936,"""2014–2020"""
2014,2020,"""Nacidos en el extranjero""",5952770,7014753,6,1061983,176997.166667,2.773752,129.146936,"""2014–2020"""
2020,2025,"""Nacidos en España""",40303297,39664087,5,-639210,-127842.0,-0.319232,-35.310651,"""2020–2025"""
2020,2025,"""Nacidos en el extranjero""",7014753,9464210,5,2449457,489891.4,6.173075,135.310651,"""2020–2025"""


In [142]:
res_C

ini,fin,grupo,pob_ini,pob_fin,años,var_abs,media_anual,tasa_anual,contribucion,periodo
i64,i64,str,i64,i64,i64,i64,f64,f64,f64,str
2002,2008,"""Españoles nacidos en España""",38621928,39537144,6,915216,152536.0,0.391103,19.751437,"""2002–2008"""
2002,2008,"""Españoles nacidos en el extran…",675371,1045500,6,370129,61688.166667,7.554917,7.987819,"""2002–2008"""
2002,2008,"""Extranjeros nacidos en España""",79245,252875,6,173630,28938.333333,21.335813,3.747139,"""2002–2008"""
2002,2008,"""Extranjeros nacidos en el extr…",1658727,4833420,6,3174693,529115.5,19.512483,68.513605,"""2002–2008"""
2008,2014,"""Españoles nacidos en España""",39537144,40109398,6,572254,95375.666667,0.239788,69.212692,"""2008–2014"""
…,…,…,…,…,…,…,…,…,…,…
2014,2020,"""Extranjeros nacidos en el extr…",4242776,4704406,6,461630,76938.333333,1.736263,56.138469,"""2014–2020"""
2020,2025,"""Españoles nacidos en España""",39766425,39164944,5,-601481,-120296.2,-0.304354,-33.22646,"""2020–2025"""
2020,2025,"""Españoles nacidos en el extran…",2310347,3051382,5,741035,148207.0,5.721636,40.935574,"""2020–2025"""


In [148]:
def grafico_anual(anual, colores, titulo, bandas=None):
    d = (anual.sort("grupo", "año")
         .with_columns(var_abs=pl.col("poblacion").diff().over("grupo")).drop_nulls("var_abs"))
    var_total = d.group_by("año").agg(var_abs=pl.col("var_abs").sum()).sort("año")
    pob_total = anual.group_by("año").agg(poblacion=pl.col("poblacion").sum()).sort("año")

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Bandas de periodos (opcional), primero para que queden al fondo
    if bandas:
        for i, (a, b) in enumerate(zip(bandas[:-1], bandas[1:])):
            if i % 2 == 0:
                fig.add_vrect(x0=a + 0.5, x1=b + 0.5, fillcolor="rgba(127,127,127,0.10)", line_width=0, layer="below")

    for g, c in colores.items():
        dg = d.filter(pl.col("grupo") == g)
        fig.add_bar(x=dg["año"], y=dg["var_abs"], name=g, marker_color=c, secondary_y=False,
                    hovertemplate="%{y:,.0f}<extra>" + g + "</extra>")

    fig.add_scatter(x=var_total["año"], y=var_total["var_abs"], name="Variación total (eje izq.)",
                    mode="markers", marker=dict(symbol="diamond", size=7, color="#111827", line=dict(color="white", width=1)),
                    hovertemplate="%{y:,.0f}<extra>Variación total</extra>", secondary_y=False)
    fig.add_scatter(x=pob_total["año"], y=pob_total["poblacion"], name="Población total a 1 de enero (eje der.)",
                    mode="lines", line=dict(color="#6b7280", width=2.5),
                    hovertemplate="%{y:,.0f}<extra>Población total</extra>", secondary_y=True)

    fig.update_layout(height=470, barmode="relative", margin=dict(r=70),
                      title=dict(text=titulo, x=0, font=dict(size=15)))
    fig.update_yaxes(title_text="Variación anual (personas)", secondary_y=False)
    fig.update_yaxes(title_text="Población", tickformat=".2s", showgrid=False, secondary_y=True)
    fig.update_xaxes(range=[2001.5, 2025.5], dtick=2)
    add_source(fig, "Fuente: INE, Estadística Continua de Población. Elaboración propia.")
    return fig

def grafico_periodos(res, colores, titulo):
    fig = new_figure(y_title="Variación media anual (personas)", height=440, barmode="relative", title=dict(text=titulo, x=0, font=dict(size=15)))
    for g, c in colores.items():
        r = res.filter(pl.col("grupo") == g)
        fig.add_bar(x=r["periodo"], y=r["media_anual"], name=g, marker_color=c,
                    text=[f"{es(v, 0)} %" for v in r["contribucion"]], textposition="inside",
                    customdata=r.select("tasa_anual", "contribucion").rows(),
                    hovertemplate="%{y:,.0f} personas/año<br>%{customdata[0]:.2f} % anual<br>%{customdata[1]:.0f} % del Δ total<extra></extra>")
    neto = res.group_by("periodo", "ini").agg(media_anual=pl.col("media_anual").sum()).sort("ini")
    fig.add_scatter(x=neto["periodo"], y=neto["media_anual"], name="Total", mode="markers",
                    marker=dict(symbol="diamond", size=9, color="#111827", line=dict(color="white", width=1)),
                    hovertemplate="%{y:,.0f} personas/año<extra>Total</extra>")
    add_source(fig, "Fuente: INE, Estadística Continua de Población. Elaboración propia.")
    return fig

In [153]:

col_A = {"Española": GREY, "Extranjera": ACCENT}
col_B = {"Nacidos en España": GREY, "Nacidos en el extranjero": ACCENT}
col_C = {"Españoles nacidos en España": GREY, "Españoles nacidos en el extranjero": PALETTE[1],
         "Extranjeros nacidos en España": PALETTE[2], "Extranjeros nacidos en el extranjero": ACCENT}

grafico_anual(por_nacionalidad, col_A, "Por nacionalidad", bandas=limites).show()
grafico_periodos(res_A, col_A, "Por nacionalidad").show()
grafico_periodos(res_B, col_B, "Por nacimiento").show()
grafico_periodos(res_C, col_C, "Cuatro grupos").show()

In [ ]:
def grafico_cuota(anual, colores, titulo, bandas=None):
    d = (anual
         .with_columns(pct=100 * pl.col("poblacion") / pl.col("poblacion").sum().over("año"))
         .sort("grupo", "año"))
    mayor = d.group_by("grupo").agg(pl.col("pct").mean()).sort("pct")["grupo"][-1]   # grupo dominante

    fig = new_figure(y_title="% de la población total", height=440,
                     title=dict(text=f"{titulo} · peso sobre la población total", x=0, font=dict(size=15)))
    if bandas:
        for i, (a, b) in enumerate(zip(bandas[:-1], bandas[1:])):
            if i % 2 == 0:
                fig.add_vrect(x0=a + 0.5, x1=b + 0.5, fillcolor="rgba(127,127,127,0.10)", line_width=0, layer="below")

    for g, c in colores.items():
        if g == mayor:
            continue
        dg = d.filter(pl.col("grupo") == g)
        etiquetas = [""] * (dg.height - 1) + [f"{es(dg['pct'][-1], 1)} %"]          # valor solo en el último punto
        fig.add_scatter(x=dg["año"], y=dg["pct"], name=g, mode="lines+markers+text",
                        line=dict(color=c, width=2.5), marker=dict(size=5),
                        text=etiquetas, textposition="middle right", textfont=dict(color=c),
                        hovertemplate="%{y:.1f} %<extra>" + g + "</extra>")

    fig.update_yaxes(ticksuffix=" %", tickformat=".0f", rangemode="tozero")
    fig.update_xaxes(range=[2001.5, 2026.5], dtick=2)
    fig.add_annotation(text=f"Resto de la población: {mayor}", xref="paper", yref="paper", x=0, y=1.0,
                       xanchor="left", yanchor="top", showarrow=False, font=dict(size=11))
    add_source(fig, "Fuente: INE, Estadística Continua de Población. Elaboración propia.")
    return fig

In [151]:
grafico_cuota(por_nacionalidad, col_A, "Por nacionalidad", bandas=limites).show()
grafico_cuota(por_nacimiento, col_B, "Por país de nacimiento", bandas=limites).show()
grafico_cuota(cuatro_grupos, col_C, "Cuatro grupos", bandas=limites).show()

In [152]:
def cuota_de(anual, grupo):
    return (anual.with_columns(pct=100 * pl.col("poblacion") / pl.col("poblacion").sum().over("año"))
                 .filter(pl.col("grupo") == grupo).sort("año"))

a = cuota_de(por_nacionalidad, "Extranjera")
b = cuota_de(por_nacimiento, "Nacidos en el extranjero")

fig = new_figure(y_title="% de la población total", height=420)
fig.add_scatter(x=b["año"], y=b["pct"], name="Nacidos en el extranjero", mode="lines+markers",
                line=dict(color=ACCENT, width=2.5), marker=dict(size=5), hovertemplate="%{y:.1f} %<extra></extra>")
fig.add_scatter(x=a["año"], y=a["pct"], name="Nacionalidad extranjera", mode="lines+markers",
                line=dict(color=PALETTE[1], width=2.5), marker=dict(size=5), hovertemplate="%{y:.1f} %<extra></extra>")
fig.update_yaxes(ticksuffix=" %", tickformat=".0f", rangemode="tozero")
fig.update_xaxes(dtick=2)
add_source(fig, "Fuente: INE, Estadística Continua de Población. Elaboración propia.")
fig.show()

### Q2.1: ¿Cuánto ha crecido la población residente desde 2002 y a qué ritmo, por periodos? --> A nivel Provincial

In [102]:
serie_total = (
    df2
    .filter(
        pl.col("Sexo") == "Total",
        pl.col("Grupo quinquenal de edad") == "Todas las edades",
        pl.col('provincia_nombre') != "Total Nacional",
        pl.col("Nacionalidad") == "Total",
        pl.col("Total").is_not_null(),
    )
    .select(
        provincia_codigo=pl.col("provincia_codigo"),
        provincia=pl.col("provincia_nombre"),
        fecha=pl.col("Periodo"),
        poblacion=pl.col("Total"),
    )
)

serie_total_anual_provincial = (
    serie_total
    .filter(pl.col("fecha").dt.month() == 1, pl.col("fecha").dt.day() == 1)
    .sort("provincia", "fecha")
    .with_columns(
        var_abs=pl.col("poblacion").diff().over("provincia"),
        var_pct=100 * pl.col("poblacion").pct_change().over("provincia"),
    )
)

In [103]:
serie_total_anual_provincial

provincia_codigo,provincia,fecha,poblacion,var_abs,var_pct
str,str,date,i64,i64,f64
"""02""","""Albacete""",2021-01-01,384913,null,null
"""02""","""Albacete""",2022-01-01,384089,-824,-0.214074
"""02""","""Albacete""",2023-01-01,385572,1483,0.386108
"""02""","""Albacete""",2024-01-01,387066,1494,0.387476
"""02""","""Albacete""",2025-01-01,388637,1571,0.405874
…,…,…,…,…,…
"""05""","""Ávila""",2022-01-01,157213,113,0.071929
"""05""","""Ávila""",2023-01-01,157865,652,0.414724
"""05""","""Ávila""",2024-01-01,158495,630,0.399075


### Q3.1: ¿Cuánto ha crecido la población residente desde 2002 y a qué ritmo, por periodos? --> A nivel Autonomico

In [106]:
CCAA_PROVINCIAS = {
    "Andalucía":          ["Almería", "Cádiz", "Córdoba", "Granada", "Huelva", "Jaén", "Málaga", "Sevilla"],
    "Aragón":             ["Huesca", "Teruel", "Zaragoza"],
    "Asturias":           ["Asturias"],
    "Illes Balears":      ["Balears, Illes"],
    "Canarias":           ["Palmas, Las", "Santa Cruz de Tenerife"],
    "Cantabria":          ["Cantabria"],
    "Castilla y León":    ["Ávila", "Burgos", "León", "Palencia", "Salamanca", "Segovia", "Soria", "Valladolid", "Zamora"],
    "Castilla-La Mancha": ["Albacete", "Ciudad Real", "Cuenca", "Guadalajara", "Toledo"],
    "Cataluña":           ["Barcelona", "Girona", "Lleida", "Tarragona"],
    "C. Valenciana":      ["Alicante/Alacant", "Castellón/Castelló", "Valencia/València"],
    "Extremadura":        ["Badajoz", "Cáceres"],
    "Galicia":            ["Coruña, A", "Lugo", "Ourense", "Pontevedra"],
    "Madrid":             ["Madrid"],
    "Murcia":             ["Murcia"],
    "Navarra":            ["Navarra"],
    "País Vasco":         ["Araba/Álava", "Bizkaia", "Gipuzkoa"],
    "La Rioja":           ["Rioja, La"],
    "Ceuta":              ["Ceuta"],
    "Melilla":            ["Melilla"],
}
PROV_A_CCAA = {prov: ccaa for ccaa, provs in CCAA_PROVINCIAS.items() for prov in provs}

In [107]:
serie_total = (
    df2
    .filter(
        pl.col("Sexo") == "Total",
        pl.col("Grupo quinquenal de edad") == "Todas las edades",
        pl.col('provincia_nombre') != "Total Nacional",
        pl.col("Nacionalidad") == "Total",
        pl.col("Total").is_not_null(),
    )
    .select(
        provincia_codigo=pl.col("provincia_codigo"),
        provincia=pl.col("provincia_nombre"),
        fecha=pl.col("Periodo"),
        poblacion=pl.col("Total"),
    )
)

serie_total_anual_autonomica = (
    serie_total
    .filter(pl.col("provincia") != "Total Nacional")          # evita doble conteo y que replace_strict falle
    .with_columns(ccaa=pl.col("provincia").replace_strict(PROV_A_CCAA))
    .group_by("ccaa", "fecha")
    .agg(poblacion=pl.col("poblacion").sum(), n_prov=pl.len())
    .filter(pl.col("fecha").dt.month() == 1, pl.col("fecha").dt.day() == 1)
    .sort("ccaa", "fecha")
    .with_columns(
        var_abs=pl.col("poblacion").diff().over("ccaa"),
        var_pct=100 * pl.col("poblacion").pct_change().over("ccaa"),
    )
)

In [108]:
serie_total_anual_autonomica

ccaa,fecha,poblacion,n_prov,var_abs,var_pct
str,date,i64,u32,i64,f64
"""Andalucía""",2021-01-01,8442979,8,null,null
"""Andalucía""",2022-01-01,8468639,8,25660,0.303921
"""Andalucía""",2023-01-01,8540401,8,71762,0.847385
"""Andalucía""",2024-01-01,8587093,8,46692,0.546719
"""Andalucía""",2025-01-01,8630679,8,43586,0.507576
…,…,…,…,…,…
"""País Vasco""",2022-01-01,2186490,3,-6977,-0.318081
"""País Vasco""",2023-01-01,2196431,3,9941,0.454656
"""País Vasco""",2024-01-01,2207226,3,10795,0.491479


---

---

In [75]:
MAPPING_PAIS = {
    "Total":                    "Total",
    "España":                   "España",
    "Extranjero":               "Extranjero (total)",
    "UE28 sin España":          "Europa",
    "Europa menos UE28":        "Europa",
    "UE27_2020 sin España":     "Europa",
    "Europa menos UE27_2020":   "Europa",
    "África":                   "África",
    "Sudamérica":               "Sudamérica",
    "Centro América y Caribe":  "Centroamérica y Caribe",
    "América del Norte":        "América del Norte",
    "Asia":                     "Asia",
    "Oceanía":                  "Oceanía",
}
AGREGADOS = ["Total", "España", "Extranjero (total)"]

tabla = (
    df1
    .filter(
        pl.col("Sexo") == "Total",
        pl.col("Edad simple") == "Todas las edades",
        pl.col("Nacionalidad").is_in(["Española", "Extranjera"]),
        pl.col("Total").is_not_null(),
    )
    .select(
        fecha=pl.col("Periodo"),
        nacionalidad=pl.col("Nacionalidad"),
        pais=pl.col("País de nacimiento"),
        poblacion=pl.col("Total"),
    )
    .filter(pl.col("pais").is_in(list(MAPPING_PAIS)))
    .with_columns(pais=pl.col("pais").replace_strict(MAPPING_PAIS))
    .group_by("fecha", "nacionalidad", "pais")
    .agg(poblacion=pl.col("poblacion").sum(), n=pl.len())
    .with_columns(
        nivel=pl.when(pl.col("pais").is_in(AGREGADOS)).then(pl.lit("agregado")).otherwise(pl.lit("región"))
    )
    .sort("fecha", "nacionalidad", "pais")
)


In [81]:
tabla.head(10)

fecha,nacionalidad,pais,poblacion,n,nivel
date,str,str,i64,u32,str
2002-01-01,"""Española""","""América del Norte""",23411,1,"""región"""
2002-01-01,"""Española""","""Asia""",21159,1,"""región"""
2002-01-01,"""Española""","""Centroamérica y Caribe""",46598,1,"""región"""
2002-01-01,"""Española""","""España""",38621928,1,"""agregado"""
2002-01-01,"""Española""","""Europa""",309416,2,"""región"""
2002-01-01,"""Española""","""Extranjero (total)""",675371,1,"""agregado"""
2002-01-01,"""Española""","""Oceanía""",3037,1,"""región"""
2002-01-01,"""Española""","""Sudamérica""",173563,1,"""región"""
2002-01-01,"""Española""","""Total""",39297299,1,"""agregado"""


In [83]:
paises = ["Total", "España", "Extranjero (total)", "Europa", "África",
          "Sudamérica", "Centroamérica y Caribe", "América del Norte", "Asia", "Oceanía"]
nacionalidades = ["Española", "Extranjera"]
colores_nac = {"Española": GREY, "Extranjera": ACCENT}
pais_inicial = "Total"

fig = new_figure(y_title="Personas", height=460)

trazas_pais = []                       # país de cada traza, en el orden en que se añaden
for p in paises:
    for n in nacionalidades:
        d = tabla.filter((pl.col("pais") == p) & (pl.col("nacionalidad") == n)).sort("fecha")
        fig.add_scatter(
            x=d["fecha"], y=d["poblacion"], name=n, legendgroup=n,
            mode="lines+markers", marker=dict(size=4),
            line=dict(color=colores_nac[n], width=2.5),
            hovertemplate="%{y:,.0f}<extra></extra>",
            visible=(p == pais_inicial),
        )
        trazas_pais.append(p)

botones = [
    dict(
        label=p, method="update",
        args=[{"visible": [tp == p for tp in trazas_pais]},
              {"title.text": f"Población según nacionalidad · nacidos en: {p}"}],
    )
    for p in paises
]

fig.update_layout(
    title=dict(text=f"Población según nacionalidad · nacidos en: {pais_inicial}", x=0, font=dict(size=15)),
    margin=dict(t=90),
    updatemenus=[dict(
        buttons=botones, direction="down", showactive=True,
        active=paises.index(pais_inicial),
        x=1, xanchor="right", y=1.02, yanchor="bottom",
    )],
)
fig.update_xaxes(dtick="M24", tickformat="%Y")
add_source(fig, "Fuente: INE, Estadística Continua de Población. Elaboración propia.")
fig.show()

In [84]:
nacionalidades = ["Extranjera", "Española"]
regiones = ["Sudamérica", "Centroamérica y Caribe", "Europa", "África", "Asia", "América del Norte", "Oceanía"]
agregados = ["Extranjero (total)", "España", "Total"]      # aparecen en la leyenda, apagados
paises = regiones + agregados

colores = {
    "Sudamérica": "#c2410c", "Centroamérica y Caribe": "#a16207", "Europa": "#2563eb",
    "África": "#0d9488", "Asia": "#be185d", "América del Norte": "#7c3aed", "Oceanía": "#65a30d",
    "Extranjero (total)": "#333333", "España": "#666666", "Total": "#999999",
}
nac_inicial = "Extranjera"

fig = new_figure(y_title="Personas", height=500)

trazas = []                                   # (nacionalidad, país) de cada traza, en orden de creación
for n in nacionalidades:
    for p in paises:
        d = tabla.filter((pl.col("pais") == p) & (pl.col("nacionalidad") == n)).sort("fecha")
        es_region = p in regiones
        fig.add_scatter(
            x=d["fecha"], y=d["poblacion"], name=p,
            mode="lines+markers", marker=dict(size=4),
            line=dict(color=colores[p], width=2.2, dash="solid" if es_region else "dash"),
            hovertemplate="%{y:,.0f}<extra></extra>",
            visible=(True if es_region else "legendonly") if n == nac_inicial else False,
        )
        trazas.append((n, p))

def visibilidad(n_sel):
    return [(True if p in regiones else "legendonly") if n == n_sel else False for n, p in trazas]

fig.update_layout(
    title=dict(text="Población por país de nacimiento", x=0, font=dict(size=15)),
    margin=dict(t=90, r=190),
    legend=dict(orientation="v", x=1.02, y=1, xanchor="left", yanchor="top"),
    updatemenus=[
        dict(buttons=[dict(label=n, method="update", args=[{"visible": visibilidad(n)}]) for n in nacionalidades],
             active=nacionalidades.index(nac_inicial), direction="down", showactive=True,
             x=0, xanchor="left", y=1.14, yanchor="bottom"),
        dict(buttons=[dict(label="Escala lineal", method="relayout", args=["yaxis.type", "linear"]),
                      dict(label="Escala log", method="relayout", args=["yaxis.type", "log"])],
             active=0, direction="down", showactive=True,
             x=0.25, xanchor="left", y=1.14, yanchor="bottom"),
    ],
)
fig.update_xaxes(dtick="M24", tickformat="%Y")
add_source(fig, "Fuente: INE, Estadística Continua de Población. Elaboración propia.")
fig.show()

## 4. Cifras y gráficos finales (→ post)

## Notas

- Decisiones tomadas y por qué.
- Dudas abiertas / limitaciones.